# Row Level Security (RLS)

## Overview
Row Level Security enforces per-row access rules at the database engine level. Regardless of what SQL a query contains, PostgreSQL silently applies the relevant policy — making it impossible for a user to see or modify rows they are not permitted to access, even with a direct `psql` connection or BI tool.

**Core DDL:**
```sql
ALTER TABLE patients ENABLE ROW LEVEL SECURITY;
ALTER TABLE patients FORCE ROW LEVEL SECURITY;  -- apply even to table owner

CREATE POLICY policy_name ON table_name
    FOR {ALL | SELECT | INSERT | UPDATE | DELETE}
    TO role_name
    USING (row_filter_expression)          -- for reads
    WITH CHECK (write_filter_expression);  -- for INSERT/UPDATE
```

**Without any policy, RLS denies all rows** — you must create explicit allow policies.

---

In [1]:
import sqlite3, hashlib, secrets, base64, hmac
from datetime import datetime

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
CREATE TABLE users (
    user_id TEXT PRIMARY KEY, username TEXT NOT NULL UNIQUE,
    role_name TEXT NOT NULL, department TEXT, active INTEGER DEFAULT 1
);
CREATE TABLE patients (
    patient_id TEXT PRIMARY KEY, full_name TEXT NOT NULL,
    dob TEXT NOT NULL, province TEXT NOT NULL, assigned_clinician TEXT
);
CREATE TABLE patient_records (
    record_id TEXT PRIMARY KEY, patient_id TEXT NOT NULL,
    record_type TEXT NOT NULL, content TEXT NOT NULL,
    sensitivity TEXT DEFAULT 'normal', created_by TEXT NOT NULL, created_at TEXT
);
CREATE TABLE financial_accounts (
    account_id TEXT PRIMARY KEY, customer_id TEXT NOT NULL,
    account_type TEXT NOT NULL, balance_enc TEXT, ssn_hash TEXT, created_at TEXT
);
CREATE TABLE audit_log (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT, event_time TEXT DEFAULT (datetime('now')),
    username TEXT NOT NULL, action TEXT NOT NULL, table_name TEXT,
    record_id TEXT, old_value TEXT, new_value TEXT, ip_address TEXT, success INTEGER DEFAULT 1
);
CREATE TABLE role_permissions (
    role_name TEXT NOT NULL, table_name TEXT NOT NULL,
    can_select INTEGER DEFAULT 0, can_insert INTEGER DEFAULT 0,
    can_update INTEGER DEFAULT 0, can_delete INTEGER DEFAULT 0,
    row_filter TEXT, PRIMARY KEY (role_name, table_name)
);
""")

conn.executemany("INSERT INTO users VALUES (?,?,?,?,?)", [
    ('U001','dr.lee',    'clinician','Cardiology',    1),
    ('U002','dr.pham',   'clinician','Endocrinology',1),
    ('U003','analyst1',  'analyst',  'Reporting',    1),
    ('U004','admin1',    'admin',    'IT',           1),
    ('U005','auditor1',  'auditor',  'Compliance',   1),
    ('U006','patient_p001','patient',None,            1),
])
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", [
    ('P001','Alice Ngata',  '1980-03-15','NB','dr.lee'),
    ('P002','Bob Chen',     '1972-07-22','ON','dr.pham'),
    ('P003','Fatima Rashid','1986-11-05','BC','dr.lee'),
    ('P004','James MacLeod','1963-01-30','NS','dr.pham'),
    ('P005','Mei-Ling Tran','1966-08-19','QC','dr.lee'),
])
conn.executemany("INSERT INTO patient_records VALUES (?,?,?,?,?,?,?)", [
    ('R001','P001','diagnosis',  'Hypertension, Stage 2',      'normal',    'dr.lee', '2024-01-15'),
    ('R002','P001','prescription','Lisinopril 10mg daily',     'normal',    'dr.lee', '2024-01-15'),
    ('R003','P002','diagnosis',  'Type 2 Diabetes',            'normal',    'dr.pham','2024-01-10'),
    ('R004','P002','lab',        'HbA1c 8.4%',                 'normal',    'dr.pham','2024-01-10'),
    ('R005','P003','diagnosis',  'Asthma, moderate persistent','sensitive', 'dr.lee', '2024-02-01'),
    ('R006','P004','diagnosis',  'CKD Stage 3, T2DM',          'sensitive', 'dr.pham','2024-02-15'),
    ('R007','P005','prescription','Insulin glargine 20u',      'restricted','dr.lee', '2024-03-01'),
])
salt = secrets.token_hex(8)
def hash_ssn(ssn): return hashlib.sha256((salt+ssn).encode()).hexdigest()
conn.executemany("INSERT INTO financial_accounts VALUES (?,?,?,?,?,datetime('now'))", [
    ('ACC001','P001','chequing','[encrypted:AES256]',hash_ssn('123-45-6789')),
    ('ACC002','P002','savings', '[encrypted:AES256]',hash_ssn('987-65-4321')),
    ('ACC003','P003','chequing','[encrypted:AES256]',hash_ssn('456-78-9012')),
])
conn.executemany("INSERT INTO role_permissions VALUES (?,?,?,?,?,?,?)", [
    ('clinician','patients',       1,0,1,0,'assigned_clinician = current_user'),
    ('clinician','patient_records',1,1,1,0,'patient_id IN (SELECT patient_id FROM patients WHERE assigned_clinician = current_user)'),
    ('analyst',  'patients',       1,0,0,0,'province IS NOT NULL'),
    ('analyst',  'patient_records',1,0,0,0,"sensitivity = 'normal'"),
    ('admin',    'patients',       1,1,1,1,None),
    ('admin',    'patient_records',1,1,1,1,None),
    ('auditor',  'audit_log',      1,0,0,0,None),
    ('auditor',  'patients',       1,0,0,0,None),
    ('patient',  'patients',       1,0,0,0,'patient_id = current_patient_id()'),
    ('patient',  'patient_records',1,0,0,0,'patient_id = current_patient_id()'),
])
conn.executemany(
    "INSERT INTO audit_log (username,action,table_name,record_id,old_value,new_value,ip_address,success) VALUES (?,?,?,?,?,?,?,?)", [
    ('dr.lee',  'SELECT','patient_records','R001',None,None,'10.0.1.5',1),
    ('dr.lee',  'UPDATE','patient_records','R002','{"content":"Lisinopril 5mg"}','{"content":"Lisinopril 10mg"}','10.0.1.5',1),
    ('analyst1','SELECT','patients',       None,  None,None,'10.0.2.8',1),
    ('analyst1','EXPORT','patient_records',None,  None,None,'10.0.2.8',1),
    ('unknown', 'LOGIN', None,             None,  None,None,'203.0.113.9',0),
    ('dr.pham', 'SELECT','patient_records','R005',None,None,'10.0.1.6',0),
    ('admin1',  'DELETE','patient_records','R007',None,None,'10.0.1.1',1),
])
conn.commit()
print("Dataset loaded:")
for t in ['users','patients','patient_records','financial_accounts','audit_log','role_permissions']:
    print(f"  {t}: {conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]} rows")

print("=== Enabling RLS and creating policies ===")
print()
print("""
-- Step 1: Enable RLS
ALTER TABLE patients ENABLE ROW LEVEL SECURITY;
ALTER TABLE patients FORCE ROW LEVEL SECURITY;  -- applies even to table owner

-- Step 2: Create policies (no policy = deny all rows)

-- Clinicians see only their own patients:
CREATE POLICY clinician_own_patients ON patients
    FOR ALL TO clinician_role
    USING (assigned_clinician = current_user)
    WITH CHECK (assigned_clinician = current_user);

-- Admins see everything:
CREATE POLICY admin_all ON patients
    FOR ALL TO admin_role
    USING (true);

-- Analysts see all rows (column masking done via view):
CREATE POLICY analyst_read ON patients
    FOR SELECT TO analyst_role
    USING (true);

-- Patient portal: each patient sees only their own record:
CREATE POLICY patient_self ON patients
    FOR SELECT TO patient_role
    USING (patient_id = current_setting('app.current_patient_id'));
""")

# Simulate RLS with Python-level row filtering
def query_as_role(role, username, table, conn):
    perm = conn.execute(
        "SELECT row_filter FROM role_permissions WHERE role_name=? AND table_name=?",
        (role, table)
    ).fetchone()
    if not perm: return []
    rf = perm[0]
    if rf and rf != '(none)':
        filter_sql = rf.replace('current_user', f"'{username}'")
        sql = f"SELECT patient_id, full_name FROM patients WHERE {filter_sql}"
    else:
        sql = "SELECT patient_id, full_name FROM patients"
    return conn.execute(sql).fetchall()

print("Simulated RLS — same query, different rows returned by role:")
for role, user in [('clinician','dr.lee'),('clinician','dr.pham'),('analyst','analyst1'),('admin','admin1')]:
    rows = query_as_role(role, user, 'patients', conn)
    ids = [r[0] for r in rows]
    print(f"  {user} ({role}): {len(rows)} rows → {ids}")


Dataset loaded:
  users: 6 rows
  patients: 5 rows
  patient_records: 7 rows
  financial_accounts: 3 rows
  audit_log: 7 rows
  role_permissions: 10 rows
=== Enabling RLS and creating policies ===


-- Step 1: Enable RLS
ALTER TABLE patients ENABLE ROW LEVEL SECURITY;
ALTER TABLE patients FORCE ROW LEVEL SECURITY;  -- applies even to table owner

-- Step 2: Create policies (no policy = deny all rows)

-- Clinicians see only their own patients:
CREATE POLICY clinician_own_patients ON patients
    FOR ALL TO clinician_role
    USING (assigned_clinician = current_user)
    WITH CHECK (assigned_clinician = current_user);

-- Admins see everything:
CREATE POLICY admin_all ON patients
    FOR ALL TO admin_role
    USING (true);

-- Analysts see all rows (column masking done via view):
CREATE POLICY analyst_read ON patients
    FOR SELECT TO analyst_role
    USING (true);

-- Patient portal: each patient sees only their own record:
CREATE POLICY patient_self ON patients
    FOR SELECT TO pati

In [2]:

print("=== USING vs WITH CHECK; sensitivity-based policies ===")
print()
clause_table = [
    ("USING (cond)",      "Filters SELECT / UPDATE / DELETE — hides non-matching rows"),
    ("WITH CHECK (cond)", "Filters INSERT / UPDATE — prevents writing non-matching rows"),
    ("FOR SELECT",        "Policy applies to SELECT only"),
    ("FOR INSERT",        "Policy applies to INSERT only (use WITH CHECK)"),
    ("FOR UPDATE",        "Policy applies to UPDATE"),
    ("FOR ALL",           "Policy applies to all DML — most common"),
]
print(f"  {'Clause':<22s}  Description")
print("  " + "-"*65)
for clause, desc in clause_table:
    print(f"  {clause:<22s}  {desc}")

print()
print("""
-- Sensitivity-based policy (analysts see normal records only):
CREATE POLICY analyst_normal_only ON patient_records
    FOR SELECT TO analyst_role
    USING (sensitivity = 'normal');

-- Clinicians see their own patients' records:
CREATE POLICY clinician_records ON patient_records
    FOR ALL TO clinician_role
    USING (
        patient_id IN (
            SELECT patient_id FROM patients
            WHERE assigned_clinician = current_user
        )
    )
    WITH CHECK (
        patient_id IN (
            SELECT patient_id FROM patients
            WHERE assigned_clinician = current_user
        )
    );
""")

# Simulate sensitivity filter
print("Effect of sensitivity policy on patient_records:")
for role, filter_sql, label in [
    ('clinician', "patient_id IN (SELECT patient_id FROM patients WHERE assigned_clinician='dr.lee')", 'dr.lee'),
    ('analyst',   "sensitivity = 'normal'",  'analyst1'),
    ('auditor',   '1=1',                     'auditor1'),
]:
    rows = conn.execute(
        f"SELECT record_id, sensitivity FROM patient_records WHERE {filter_sql}"
    ).fetchall()
    print(f"  {label}: {len(rows)} records [{', '.join(r[0] for r in rows)}]")

print()
print("=== Application context with current_setting() ===")
print("""
-- Set context at start of each request (Python + psycopg2):
cur.execute("SET app.current_user_id = %s", (user_id,))
cur.execute("SET app.current_role    = %s", (user_role,))

-- RLS policy references session variable:
CREATE POLICY context_policy ON patient_records
    FOR ALL
    USING (
        CASE current_setting('app.current_role', true)
            WHEN 'clinician' THEN patient_id IN (
                SELECT patient_id FROM patients
                WHERE assigned_clinician = current_setting('app.current_user_id', true)
            )
            WHEN 'admin'    THEN true
            WHEN 'analyst'  THEN sensitivity = 'normal'
            ELSE false
        END
    );
""")


=== USING vs WITH CHECK; sensitivity-based policies ===

  Clause                  Description
  -----------------------------------------------------------------
  USING (cond)            Filters SELECT / UPDATE / DELETE — hides non-matching rows
  WITH CHECK (cond)       Filters INSERT / UPDATE — prevents writing non-matching rows
  FOR SELECT              Policy applies to SELECT only
  FOR INSERT              Policy applies to INSERT only (use WITH CHECK)
  FOR UPDATE              Policy applies to UPDATE
  FOR ALL                 Policy applies to all DML — most common


-- Sensitivity-based policy (analysts see normal records only):
CREATE POLICY analyst_normal_only ON patient_records
    FOR SELECT TO analyst_role
    USING (sensitivity = 'normal');

-- Clinicians see their own patients' records:
CREATE POLICY clinician_records ON patient_records
    FOR ALL TO clinician_role
    USING (
        patient_id IN (
            SELECT patient_id FROM patients
            WHERE assign

---
## Common Pitfalls

**1. Forgetting FORCE ROW LEVEL SECURITY for the table owner**
`ENABLE ROW LEVEL SECURITY` applies RLS to non-owner roles only. The table owner bypasses all policies unless you also run `ALTER TABLE t FORCE ROW LEVEL SECURITY`. If the application connects as the table owner, it silently sees all rows.

**2. No policy = no access (not all access)**
When RLS is enabled and no policy matches, that role sees zero rows. This is correct but surprises developers expecting unrestricted access. Always create an explicit `USING (true)` policy for admin roles.

**3. RLS policies with correlated subqueries on large tables**
`USING (patient_id IN (SELECT patient_id FROM patients WHERE assigned_clinician = current_user))` is evaluated per-row on every query. On large tables this becomes expensive. Materialise the clinician's patient list in a session variable or use a JOIN.

**4. Using current_user in policies when all app users share one DB role**
If all connections use `app_user`, `current_user` is always `app_user` — the policy has no effect. Pass the real user identity via `current_setting('app.user_id', true)` set by the application at session start.

**5. SECURITY DEFINER views bypassing RLS**
A view created with `SECURITY DEFINER` runs as its owner and bypasses RLS on underlying tables. Use `SECURITY INVOKER` (PostgreSQL 15+) or ensure the view owner has appropriate policies enforced.

**6. Missing WITH CHECK on INSERT/UPDATE policies**
A policy with only `USING (assigned_clinician = current_user)` allows a clinician to insert a patient assigned to a different clinician — USING doesn't cover INSERT. Add `WITH CHECK (assigned_clinician = current_user)` to enforce the constraint on writes.

---
*sql_methods_library - Samantha McGarrigle*